# Ecological Overshoot Analysis

Top 10 and bottom 10 countries across key ecological footprint and biocapacity metrics, with visualizations.

Data: `data/overshoot_results.csv` and `data/overshoot_global_summary.csv` generated by `scripts/calculate_overshoot.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

ROOT = Path("__file__").resolve().parent if "__file__" in dir() else Path(".").resolve().parent
DATA = ROOT / "data"

df = pd.read_csv(DATA / "overshoot_results.csv")
gs = pd.read_csv(DATA / "overshoot_global_summary.csv")

# Add derived columns
df["deficit_per_capita"] = df["ecological_deficit_gha"] / (df["population"] * 1000)

# Latest year slice (for rankings)
YEAR = df["year"].max()
latest = df[df["year"] == YEAR].copy()

# Filter out very small territories (pop < 10k = 10 in thousands) for per-capita rankings
latest_big = latest[latest["population"] > 10].copy()

print(f"Data: {len(df)} rows, {df['area_code'].nunique()} countries, years {df['year'].min()}-{df['year'].max()}")
print(f"Rankings based on: {YEAR}  (countries with pop > 10,000)")

---
## 1. Global Overshoot Trend

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1a: World EF vs BC
ax = axes[0]
ax.plot(gs["year"], gs["world_ef_gha"] / 1e9, "o-", color="#d62728", linewidth=2, label="Ecological Footprint")
ax.plot(gs["year"], gs["world_bc_gha"] / 1e9, "s-", color="#2ca02c", linewidth=2, label="Biocapacity")
ax.fill_between(gs["year"], gs["world_bc_gha"] / 1e9, gs["world_ef_gha"] / 1e9, alpha=0.15, color="#d62728")
ax.set_ylabel("Billion global hectares")
ax.set_title("World EF vs Biocapacity")
ax.legend()

# 1b: Number of Earths
ax = axes[1]
ax.bar(gs["year"], gs["number_of_earths"], color="#ff7f0e", edgecolor="white")
ax.axhline(1.0, color="#2ca02c", linestyle="--", linewidth=1.5, label="1 Earth")
ax.set_ylabel("Number of Earths")
ax.set_title("Number of Earths Required")
ax.legend()

# 1c: Overshoot Day
ax = axes[2]
from datetime import datetime, timedelta
dates = [datetime(int(y), 1, 1) + timedelta(days=int(d) - 1) for y, d in zip(gs["year"], gs["overshoot_day"])]
day_of_year = gs["overshoot_day"].values
ax.plot(gs["year"], day_of_year, "D-", color="#9467bd", linewidth=2, markersize=8)
ax.set_ylabel("Day of year")
ax.set_title("Earth Overshoot Day")
ax.invert_yaxis()
# Add month labels on right
for y, d in zip(gs["year"], dates):
    ax.annotate(d.strftime("%b %d"), (y, int(d.strftime("%j"))), fontsize=8, ha="center", va="bottom")

for ax in axes:
    ax.set_xlabel("Year")

fig.suptitle("Global Ecological Overshoot (2014-2023)", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---
## 2. World Footprint Composition (2023)

In [ ]:
components = ["cropland", "grazing", "forest", "fishing", "built_up", "carbon"]
labels = ["Cropland", "Grazing", "Forest", "Fishing", "Built-up", "Carbon"]
colors_ef = ["#2ca02c", "#8c564b", "#1f77b4", "#17becf", "#7f7f7f", "#d62728"]
colors_bc = ["#2ca02c", "#8c564b", "#1f77b4", "#17becf", "#7f7f7f"]

w = latest.drop(columns=["area_code", "area", "year"], errors="ignore")
ef_vals = [w[f"ef_{c}_gha"].sum() / 1e9 for c in components]
bc_components = [c for c in components if c != "carbon"]
bc_vals = [w[f"bc_{c}_gha"].sum() / 1e9 for c in bc_components]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# EF pie
ax = axes[0]
wedges, texts, autotexts = ax.pie(ef_vals, labels=labels, colors=colors_ef, autopct="%1.0f%%",
                                   startangle=90, pctdistance=0.8)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title(f"Ecological Footprint\n{sum(ef_vals):.1f} bn gha", fontweight="bold")

# BC pie
ax = axes[1]
bc_labels = [l for l in labels if l != "Carbon"]
wedges, texts, autotexts = ax.pie(bc_vals, labels=bc_labels, colors=colors_bc, autopct="%1.0f%%",
                                   startangle=90, pctdistance=0.8)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title(f"Biocapacity\n{sum(bc_vals):.1f} bn gha", fontweight="bold")

fig.suptitle(f"World Footprint Composition ({YEAR})", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---
## 3. Top 10 & Bottom 10: Ecological Footprint per Capita

In [ ]:
def plot_top_bottom(data, col, title, unit="gha/person", top_color="#d62728", bot_color="#2ca02c", n=10):
    """Horizontal bar chart showing top N and bottom N countries."""
    top = data.nlargest(n, col)[["area", col]].reset_index(drop=True)
    bot = data[data[col] > 0].nsmallest(n, col)[["area", col]].reset_index(drop=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    ax.barh(top["area"][::-1], top[col][::-1], color=top_color, edgecolor="white")
    ax.set_xlabel(unit)
    ax.set_title(f"Top {n} (highest)")
    for i, v in enumerate(top[col][::-1]):
        ax.text(v + max(top[col]) * 0.01, i, f"{v:.2f}", va="center", fontsize=9)

    ax = axes[1]
    ax.barh(bot["area"][::-1], bot[col][::-1], color=bot_color, edgecolor="white")
    ax.set_xlabel(unit)
    ax.set_title(f"Bottom {n} (lowest)")
    for i, v in enumerate(bot[col][::-1]):
        ax.text(v + max(bot[col]) * 0.02, i, f"{v:.2f}", va="center", fontsize=9)

    fig.suptitle(f"{title} ({YEAR})", fontsize=13, fontweight="bold", y=1.02)
    fig.tight_layout()
    plt.show()

    return top, bot


top_ef, bot_ef = plot_top_bottom(latest_big, "ef_per_capita_gha",
                                  "Ecological Footprint per Capita")

In [ ]:
print("Top 10 EF per capita:")
display(top_ef.style.format({"ef_per_capita_gha": "{:.2f}"}))
print("\nBottom 10 EF per capita:")
display(bot_ef.style.format({"ef_per_capita_gha": "{:.2f}"}))

---
## 4. Top 10 & Bottom 10: Biocapacity per Capita

In [ ]:
top_bc, bot_bc = plot_top_bottom(latest_big, "bc_per_capita_gha",
                                  "Biocapacity per Capita",
                                  top_color="#2ca02c", bot_color="#d62728")

In [ ]:
print("Top 10 BC per capita (most biocapacity):")
display(top_bc.style.format({"bc_per_capita_gha": "{:.2f}"}))
print("\nBottom 10 BC per capita (least biocapacity):")
display(bot_bc.style.format({"bc_per_capita_gha": "{:.2f}"}))

---
## 5. Top 10 & Bottom 10: Ecological Deficit per Capita

Negative = deficit (consuming more than available). Positive = reserve.

In [ ]:
# Largest deficits (most negative)
worst_deficit = latest_big.nsmallest(10, "deficit_per_capita")[["area", "deficit_per_capita"]].reset_index(drop=True)
# Largest reserves (most positive)
best_reserve = latest_big.nlargest(10, "deficit_per_capita")[["area", "deficit_per_capita"]].reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
bars = ax.barh(worst_deficit["area"][::-1], worst_deficit["deficit_per_capita"][::-1],
               color="#d62728", edgecolor="white")
ax.set_xlabel("gha/person")
ax.set_title("10 Largest Ecological Deficits")
ax.axvline(0, color="black", linewidth=0.8)
for i, v in enumerate(worst_deficit["deficit_per_capita"][::-1]):
    ax.text(v - 0.3, i, f"{v:.1f}", va="center", ha="right", fontsize=9, color="white", fontweight="bold")

ax = axes[1]
ax.barh(best_reserve["area"][::-1], best_reserve["deficit_per_capita"][::-1],
        color="#2ca02c", edgecolor="white")
ax.set_xlabel("gha/person")
ax.set_title("10 Largest Ecological Reserves")
ax.axvline(0, color="black", linewidth=0.8)
for i, v in enumerate(best_reserve["deficit_per_capita"][::-1]):
    ax.text(v + max(best_reserve["deficit_per_capita"]) * 0.01, i, f"+{v:.1f}", va="center", fontsize=9)

fig.suptitle(f"Ecological Deficit per Capita ({YEAR})", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
print("10 largest deficits per capita:")
display(worst_deficit.style.format({"deficit_per_capita": "{:.2f}"}))
print("\n10 largest reserves per capita:")
display(best_reserve.style.format({"deficit_per_capita": "{:+.2f}"}))

---
## 6. Top 10 & Bottom 10: Total Ecological Footprint (absolute)

In [ ]:
top_abs = latest_big.nlargest(10, "ef_total_gha")[["area", "ef_total_gha"]].reset_index(drop=True)
bot_abs = latest_big[latest_big["ef_total_gha"] > 0].nsmallest(10, "ef_total_gha")[["area", "ef_total_gha"]].reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.barh(top_abs["area"][::-1], top_abs["ef_total_gha"][::-1] / 1e9, color="#d62728", edgecolor="white")
ax.set_xlabel("Billion gha")
ax.set_title("Top 10 (largest total footprint)")
for i, v in enumerate(top_abs["ef_total_gha"][::-1] / 1e9):
    ax.text(v + 0.02, i, f"{v:.2f}", va="center", fontsize=9)

ax = axes[1]
ax.barh(bot_abs["area"][::-1], bot_abs["ef_total_gha"][::-1] / 1e6, color="#2ca02c", edgecolor="white")
ax.set_xlabel("Million gha")
ax.set_title("Bottom 10 (smallest total footprint)")
for i, v in enumerate(bot_abs["ef_total_gha"][::-1] / 1e6):
    ax.text(v + 0.02, i, f"{v:.1f}", va="center", fontsize=9)

fig.suptitle(f"Total Ecological Footprint ({YEAR})", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---
## 7. Component Breakdown: Top 10 Footprint Countries

In [ ]:
top10 = latest_big.nlargest(10, "ef_total_gha").copy()
comp_cols = [f"ef_{c}_gha" for c in components]

fig, ax = plt.subplots(figsize=(14, 6))

x = range(len(top10))
bottom = np.zeros(len(top10))
for col, label, color in zip(comp_cols, labels, colors_ef):
    vals = top10[col].values / 1e9
    ax.bar(x, vals, bottom=bottom, label=label, color=color, edgecolor="white", width=0.7)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(top10["area"].values, rotation=35, ha="right")
ax.set_ylabel("Billion gha")
ax.set_title(f"Footprint Component Breakdown — Top 10 Countries ({YEAR})", fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

---
## 8. Component Breakdown: Top 10 per Capita

In [ ]:
top10_pc = latest_big.nlargest(10, "ef_per_capita_gha").copy()

fig, ax = plt.subplots(figsize=(14, 6))

x = range(len(top10_pc))
bottom = np.zeros(len(top10_pc))
pop = top10_pc["population"].values * 1000  # convert to actual people
for col, label, color in zip(comp_cols, labels, colors_ef):
    vals = top10_pc[col].values / pop
    ax.bar(x, vals, bottom=bottom, label=label, color=color, edgecolor="white", width=0.7)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(top10_pc["area"].values, rotation=35, ha="right")
ax.set_ylabel("gha per person")
ax.set_title(f"Per-Capita Footprint Breakdown — Top 10 Countries ({YEAR})", fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

---
## 9. Top 10 & Bottom 10: Carbon Footprint per Capita

In [ ]:
latest_big["carbon_per_capita"] = latest_big["ef_carbon_gha"] / (latest_big["population"] * 1000)

top_carbon, bot_carbon = plot_top_bottom(latest_big, "carbon_per_capita",
                                          "Carbon Footprint per Capita",
                                          top_color="#d62728", bot_color="#2ca02c")

---
## 10. Top 10 & Bottom 10: Cropland Footprint per Capita

In [ ]:
latest_big["cropland_per_capita"] = latest_big["ef_cropland_gha"] / (latest_big["population"] * 1000)

top_crop, bot_crop = plot_top_bottom(latest_big, "cropland_per_capita",
                                      "Cropland Footprint per Capita",
                                      top_color="#8c564b", bot_color="#2ca02c")

---
## 11. Top 10 & Bottom 10: Fishing Footprint per Capita

In [ ]:
latest_big["fishing_per_capita"] = latest_big["ef_fishing_gha"] / (latest_big["population"] * 1000)

top_fish, bot_fish = plot_top_bottom(latest_big, "fishing_per_capita",
                                      "Fishing Grounds Footprint per Capita",
                                      top_color="#17becf", bot_color="#2ca02c")

---
## 12. Top 10 & Bottom 10: Forest Footprint per Capita

In [ ]:
latest_big["forest_per_capita"] = latest_big["ef_forest_gha"] / (latest_big["population"] * 1000)

top_forest, bot_forest = plot_top_bottom(latest_big, "forest_per_capita",
                                          "Forest Product Footprint per Capita",
                                          top_color="#1f77b4", bot_color="#2ca02c")

---
## 13. EF vs BC Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

sc = ax.scatter(latest_big["bc_per_capita_gha"],
                latest_big["ef_per_capita_gha"],
                s=latest_big["population"] / 50,
                alpha=0.6, edgecolors="white", linewidth=0.5,
                c=latest_big["deficit_per_capita"],
                cmap="RdYlGn", vmin=-10, vmax=10)

# Diagonal line: EF = BC
lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, lim], [0, lim], "k--", alpha=0.4, label="EF = BC (sustainability line)")

# Label major countries
highlight = ["United States of America", "China", "India", "Brazil", "Russia",
             "Germany", "Japan", "Indonesia", "Australia", "Canada", "Nigeria"]
for _, row in latest_big[latest_big["area"].isin(highlight)].iterrows():
    name = row["area"].replace("United States of America", "USA")\
                       .replace("Russian Federation", "Russia")
    ax.annotate(name, (row["bc_per_capita_gha"], row["ef_per_capita_gha"]),
                fontsize=8, ha="left", va="bottom",
                xytext=(5, 3), textcoords="offset points")

ax.set_xlabel("Biocapacity per capita (gha/person)")
ax.set_ylabel("Ecological Footprint per capita (gha/person)")
ax.set_title(f"EF vs Biocapacity per Capita ({YEAR})\nBubble size = population, color = deficit",
             fontweight="bold")
ax.set_xlim(0, 20)
ax.set_ylim(0, 20)
ax.legend(loc="upper left")
plt.colorbar(sc, ax=ax, label="Deficit per capita (gha)", shrink=0.7)
fig.tight_layout()
plt.show()

---
## 14. Time Series: Major Economies

In [ ]:
major = ["United States of America", "China", "India", "Brazil",
         "Russian Federation", "Germany", "Japan", "Indonesia"]
short_names = {"United States of America": "USA", "Russian Federation": "Russia"}

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for country, ax in zip(major[:4], axes.flat):
    cdf = df[df["area"] == country].sort_values("year")
    name = short_names.get(country, country)
    ax.plot(cdf["year"], cdf["ef_per_capita_gha"], "o-", color="#d62728", label="EF/cap")
    ax.plot(cdf["year"], cdf["bc_per_capita_gha"], "s-", color="#2ca02c", label="BC/cap")
    ax.fill_between(cdf["year"], cdf["bc_per_capita_gha"], cdf["ef_per_capita_gha"],
                    where=cdf["ef_per_capita_gha"] > cdf["bc_per_capita_gha"],
                    alpha=0.15, color="#d62728")
    ax.fill_between(cdf["year"], cdf["bc_per_capita_gha"], cdf["ef_per_capita_gha"],
                    where=cdf["ef_per_capita_gha"] <= cdf["bc_per_capita_gha"],
                    alpha=0.15, color="#2ca02c")
    ax.set_title(name, fontweight="bold")
    ax.set_ylabel("gha/person")
    ax.legend(fontsize=9)

fig.suptitle("EF vs BC per Capita Over Time — Major Economies", fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for country, ax in zip(major[4:], axes.flat):
    cdf = df[df["area"] == country].sort_values("year")
    name = short_names.get(country, country)
    ax.plot(cdf["year"], cdf["ef_per_capita_gha"], "o-", color="#d62728", label="EF/cap")
    ax.plot(cdf["year"], cdf["bc_per_capita_gha"], "s-", color="#2ca02c", label="BC/cap")
    ax.fill_between(cdf["year"], cdf["bc_per_capita_gha"], cdf["ef_per_capita_gha"],
                    where=cdf["ef_per_capita_gha"] > cdf["bc_per_capita_gha"],
                    alpha=0.15, color="#d62728")
    ax.fill_between(cdf["year"], cdf["bc_per_capita_gha"], cdf["ef_per_capita_gha"],
                    where=cdf["ef_per_capita_gha"] <= cdf["bc_per_capita_gha"],
                    alpha=0.15, color="#2ca02c")
    ax.set_title(name, fontweight="bold")
    ax.set_ylabel("gha/person")
    ax.legend(fontsize=9)

fig.suptitle("EF vs BC per Capita Over Time — Major Economies (cont.)", fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

---
## 15. Debtors vs Creditors: All Countries

In [ ]:
n_deficit = (latest_big["ecological_deficit_gha"] < 0).sum()
n_reserve = (latest_big["ecological_deficit_gha"] >= 0).sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie: debtor vs creditor count
ax = axes[0]
ax.pie([n_deficit, n_reserve], labels=[f"Deficit ({n_deficit})", f"Reserve ({n_reserve})"],
       colors=["#d62728", "#2ca02c"], autopct="%1.0f%%", startangle=90, textprops={"fontsize": 12})
ax.set_title(f"Countries in Deficit vs Reserve ({YEAR})", fontweight="bold")

# Histogram of deficit per capita
ax = axes[1]
vals = latest_big["deficit_per_capita"].clip(-15, 15)
ax.hist(vals, bins=40, color="#1f77b4", edgecolor="white", alpha=0.8)
ax.axvline(0, color="black", linewidth=1.2, linestyle="--")
ax.set_xlabel("Ecological deficit per capita (gha/person)")
ax.set_ylabel("Number of countries")
ax.set_title("Distribution of Ecological Deficit per Capita", fontweight="bold")
ax.annotate("DEFICIT", xy=(-10, ax.get_ylim()[1] * 0.8), fontsize=11, color="#d62728", fontweight="bold")
ax.annotate("RESERVE", xy=(3, ax.get_ylim()[1] * 0.8), fontsize=11, color="#2ca02c", fontweight="bold")

fig.tight_layout()
plt.show()

print(f"{n_deficit} countries in ecological deficit, {n_reserve} with ecological reserve")

---
## 16. Summary Tables

In [ ]:
print(f"\n{'='*70}")
print(f"GLOBAL SUMMARY ({YEAR})")
print(f"{'='*70}")
row = gs[gs["year"] == YEAR].iloc[0]
print(f"  World EF:          {row['world_ef_gha']/1e9:>8.2f} billion gha")
print(f"  World BC:          {row['world_bc_gha']/1e9:>8.2f} billion gha")
print(f"  Overshoot:         {row['overshoot_gha']/1e9:>8.2f} billion gha")
print(f"  Number of Earths:  {row['number_of_earths']:>8.2f}")
from datetime import datetime, timedelta
overshoot_date = datetime(YEAR, 1, 1) + timedelta(days=int(row['overshoot_day']) - 1)
print(f"  Overshoot Day:     {overshoot_date.strftime('%B %d')} (day {int(row['overshoot_day'])})")
print()

In [ ]:
print("Global Summary — All Years:")
display(gs.style.format({
    "world_ef_gha": "{:.2e}",
    "world_bc_gha": "{:.2e}",
    "overshoot_gha": "{:.2e}",
    "number_of_earths": "{:.3f}",
    "overshoot_day": "{:.0f}"
}))